In [9]:

from google.colab import userdata
from huggingface_hub import HfApi

token = userdata.get("HF_TOKEN")

api = HfApi(token=token)

user = api.whoami()
print("Hugging Face authentication: SUCCESS")
print("Dataset access: checking...")

repo = api.repo_info(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset"
)

print("Dataset:", repo.id)
print("Connection successful!")


Hugging Face authentication: SUCCESS
Dataset access: checking...
Dataset: FlyRank/internship-warehouse
Connection successful!


In [10]:

%pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

# Read the private token securely from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()

# Configure Hugging Face access
con.execute("INSTALL httpfs; LOAD httpfs;")

con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN ?
    )
""", [hf_token])

# March 2026 - development month
REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("DuckDB configured successfully!")


DuckDB configured successfully!


In [11]:

# Query 1: Verify row count and date range

result = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {REL}
""").df()

display(result)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [12]:

# Inspect warehouse schema without loading all rows

schema = con.sql(f"""
    DESCRIBE SELECT * FROM {REL}
""").df()

display(schema)


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [13]:

# Query 2: Verify the grain of the warehouse data

grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (
            client_hash_id,
            content_hash_id,
            report_date
        )) AS unique_page_days
    FROM {REL}
""").df()

display(grain_check)

total = int(grain_check["total_rows"].iloc[0])
unique = int(grain_check["unique_page_days"].iloc[0])

print("Duplicate page-day rows:", total - unique)
print("Grain verified:", total == unique)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_page_days
0,9841378,9841378


Duplicate page-day rows: 0
Grain verified: True


In [14]:

# Query 3: Check data availability using IS TRUE

availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM {REL}
""").df()

display(availability)

total = int(availability["total_rows"].iloc[0])
both = int(availability["both_available_rows"].iloc[0])

print(f"Both data sources available: {both:,} / {total:,}")
print(f"Availability rate: {both / total * 100:.2f}%")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


Both data sources available: 364,347 / 9,841,378
Availability rate: 3.70%


In [15]:

# ML-04: Build five features from March 2026

import numpy as np
import pandas as pd
from IPython.display import display

# Use only rows with available GSC data.
# Limit to 10,000 rows for a lightweight demonstration.
feature_df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM {REL}
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
      AND gsc_clicks IS NOT NULL
      AND gsc_avg_position IS NOT NULL
    ORDER BY report_date, client_hash_id, content_hash_id
    LIMIT 10000
""").df()

# Five features
feature_df["f_impressions"] = feature_df["gsc_impressions"]
feature_df["f_clicks"] = feature_df["gsc_clicks"]
feature_df["f_avg_position"] = feature_df["gsc_avg_position"]

feature_df["f_ctr"] = (
    feature_df["gsc_clicks"] /
    feature_df["gsc_impressions"]
)

feature_df["f_log_impressions"] = np.log1p(
    feature_df["gsc_impressions"]
)

features = [
    "f_impressions",
    "f_clicks",
    "f_avg_position",
    "f_ctr",
    "f_log_impressions"
]

print("ML-04: Five-feature frame")
print("Unit: one client-page-day record")
print("Rows:", len(feature_df))
print("Number of features:", len(features))

display(feature_df[
    ["content_hash_id", "report_date"] + features
].head(10))

print("\nFeature statistics:")
display(feature_df[features].describe().round(3))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

ML-04: Five-feature frame
Unit: one client-page-day record
Rows: 10000
Number of features: 5


,content_hash_id,report_date,f_impressions,f_clicks,f_avg_position,f_ctr,f_log_impressions
0,content_04c67f3541177192,2026-03-01,6,0,6.166667,0.00,1.945910
1,content_05acc92c165f4386,2026-03-01,7,0,8.714286,0.00,2.079442
2,content_0f30e04e709c7b5d,2026-03-01,4,0,8.750000,0.00,1.609438
3,content_1207efddce873942,2026-03-01,15,0,20.866667,0.00,2.772589
4,content_167472cd0802a8f3,2026-03-01,3,0,10.666667,0.00,1.386294
5,content_37385a4c08d8a663,2026-03-01,2,0,6.000000,0.00,1.098612
6,content_37952b007ab057b3,2026-03-01,28,0,11.178571,0.00,3.367296
7,content_3cf5722aa6fd767f,2026-03-01,1,0,8.000000,0.00,0.693147
8,content_7beb639d1052e49e,2026-03-01,37,0,7.513514,0.00,3.637586
9,content_9d5a482ec16ea617,2026-03-01,4,1,9.250000,0.25,1.609438



Feature statistics:


,f_impressions,f_clicks,f_avg_position,f_ctr,f_log_impressions
count,10000.000,10000.000,10000.000,10000.000,10000.000
mean,19.045,0.087,20.979,0.004,1.807
std,107.607,0.780,23.533,0.041,1.167
min,1.000,0.000,0.000,0.000,0.693
25%,1.000,0.000,6.000,0.000,0.693
50%,4.000,0.000,10.000,0.000,1.609
75%,9.000,0.000,24.250,0.000,2.303
max,8092.000,50.000,177.000,1.000,8.999



### Five Features and Availability

**Decision moment:** After the daily GSC data sync has completed.

**Unit of analysis:** One anonymized client-page-day observation.

1. **f_impressions:** The number of search impressions recorded for the completed day. Available after the daily data sync.

2. **f_clicks:** The number of search clicks for the completed day. Available after the daily data sync.

3. **f_avg_position:** The average search position observed during the completed day. Available after the daily data sync.

4. **f_ctr:** Calculated as clicks divided by impressions. Both values are available after the daily data sync.

5. **f_log_impressions:** A logarithmic transformation of the day's impression count. It uses only previously available impressions.

### Important Limitation

These five features describe historical search performance, not the direct effects of refreshing content.

Some features are mathematically related, so they do not represent five independent signals.

The current feature frame is a small illustrative subset of March 2026 data. A production-ready model would require appropriate historical windows, additional content information, and validation against future outcomes.

No future outcome or label-derived information is included in these five features.


In [16]:

# ML-04: Demonstrating target leakage
# This is a teaching example, NOT a validated SEO model.

import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

experiment = feature_df.copy()

# Illustrative proxy: whether the observed day had any clicks.
# This is NOT a future content-refresh outcome.
experiment["proxy_label"] = (
    experiment["gsc_clicks"] > 0
).astype(int)

# Honest predictors for this demonstration:
# Exclude clicks and CTR because they reveal the proxy.
honest_features = [
    "f_impressions",
    "f_avg_position",
    "f_log_impressions"
]

# Deliberately leaked feature: exact copy of the label.
experiment["leaked_label_feature"] = experiment["proxy_label"]

X_train, X_test, y_train, y_test = train_test_split(
    experiment,
    experiment["proxy_label"],
    test_size=0.25,
    random_state=42,
    stratify=experiment["proxy_label"]
)

def evaluate(columns):
    model = DecisionTreeClassifier(
        max_depth=4,
        random_state=42
    )
    model.fit(X_train[columns], y_train)
    predictions = model.predict(X_test[columns])
    return accuracy_score(y_test, predictions)

honest_accuracy = evaluate(honest_features)

leaked_accuracy = evaluate(
    honest_features + ["leaked_label_feature"]
)

print("DATA LEAKAGE EXPERIMENT")
print("------------------------")
print(f"Honest accuracy: {honest_accuracy:.4f}")
print(f"Leaked accuracy: {leaked_accuracy:.4f}")

# Remove the deliberately leaked column.
experiment = experiment.drop(columns=["leaked_label_feature"])

print("\nLeakage feature removed:",
      "leaked_label_feature" not in experiment.columns)

print("\nConclusion:")
print(
    "A label-derived feature can produce an artificially "
    "perfect score. The leaked feature must be removed "
    "before any meaningful evaluation."
)


DATA LEAKAGE EXPERIMENT
------------------------
Honest accuracy: 0.9580
Leaked accuracy: 1.0000

Leakage feature removed: True

Conclusion:
A label-derived feature can produce an artificially perfect score. The leaked feature must be removed before any meaningful evaluation.



### Target Leakage Experiment

I created an illustrative binary proxy indicating whether a page received at least one click on the observed day.

I first trained a decision tree using features that did not directly reveal this proxy.

Next, I intentionally added a column containing an exact copy of the target.

This leaked feature gave the model direct access to the answer, making its evaluation artificially strong.

After demonstrating the effect, I removed the leaked column.

This experiment does not measure future content-refresh success. It demonstrates why target-derived information must never be included among predictors during legitimate model evaluation.

The honest model score is also only an illustrative benchmark, not a validated content-refresh recommendation system.


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amarkosovac11/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


### Data Contract — Unit and Time Window

**Selected lane:** Refresh / Content Opportunity Scoring

**Data source:** FlyRank Internship Warehouse

**Table:** `fact_content_daily_performance`

**Unit of analysis:** One anonymized client, one content page, and one reporting day.

**Unique key:** `client_hash_id`, `content_hash_id`, and `report_date`.

**Development window:** March 1–31, 2026.

**Modeling objective:** Prioritize web pages for manual content-refresh reviews using historical search-performance signals.

**Output:** A ranked list of pages with estimated review-priority scores.

**Decision:** An SEO specialist selects which pages to investigate for potential content updates.

**Deliberate exclusion:** Future outcome information, label-derived features, and unavailable GSC records.

**Observed data:** The March 2026 table contains 9,841,378 records spanning March 1 through March 31.

This month is used for exploration and feature development. June 2026 will remain reserved for later evaluation.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


### Field Classification

**Features — five selected:**

1. `f_impressions`: Search impressions from a completed reporting day.
2. `f_clicks`: Search clicks from the completed day.
3. `f_avg_position`: Average search position for the completed day.
4. `f_ctr`: Clicks divided by impressions.
5. `f_log_impressions`: Logarithmic transformation of impressions.

These features are available after the corresponding reporting day's data sync. They cannot be used to predict an outcome for that same day if the prediction is supposed to occur before the data becomes available.

**Label / proxy:**

The ideal target would represent an independently observed future content-refresh outcome. Such a target has not yet been established in this notebook.

For the leakage demonstration, I use an illustrative binary proxy indicating whether the observed day recorded at least one click. This is not a valid measure of future refresh benefit.

**Context fields:**

- `client_hash_id`: Anonymized client identifier.
- `content_hash_id`: Anonymized content identifier.
- `report_date`: Observation date.
- `gsc_data_available`: Data availability flag.

These fields identify observations or determine eligibility but are not included in the five-feature modeling frame.

**Excluded fields:**

- `ga4_pageviews`, `ga4_sessions`, and other GA4 metrics: Excluded to avoid requiring GA4 availability for this GSC-focused analysis.
- `leaked_label_feature`: Excluded because it is derived directly from the target.
- Future observations and outcome columns: Excluded to prevent temporal leakage.
- Rows without valid GSC observations: Excluded from the feature demonstration.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


### Warehouse Verification — March 2026

Three SQL verification queries were performed using DuckDB and the FlyRank warehouse.

**Query 1 — Row count and date span**

- Total records: 9,841,378
- First reporting date: 2026-03-01
- Last reporting date: 2026-03-31

**Query 2 — Grain verification**

The proposed unique key is the combination of client identifier, content identifier, and reporting date.

The query compares the total number of records with the number of distinct key combinations.

The actual query output must be checked before concluding whether duplicates exist.

**Query 3 — Data availability**

- Total records: 9,841,378
- GSC available: 3,611,061
- GA4 available: 413,966
- Both GSC and GA4 available: 364,347
- Both sources available: approximately 3.70%

Availability was checked using SQL `IS TRUE` conditions.

**Feature frame**

A demonstration frame containing 10,000 eligible client-page-day observations was created with exactly five features.

**Leakage experiment**

- Honest accuracy: 0.9580
- Accuracy with deliberately leaked target: 1.0000
- Leaked feature removed after demonstration: Yes

The experiment demonstrates how label-derived information can artificially inflate evaluation scores. The accuracy values are not evidence of future SEO improvement.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*


### Data Limitations

**1. Incomplete data availability**

Only 3,611,061 of 9,841,378 March records have available GSC data. Only 364,347 records have both GSC and GA4 available.

Therefore, requiring both sources would exclude most observations and could introduce selection bias.

**2. Observational data**

Historical search-performance patterns do not establish that refreshing content causes improved rankings or traffic.

**3. Limited temporal coverage**

This notebook examines March 2026. Results from one month may not generalize to other periods or websites.

June 2026 is reserved for future evaluation and must not be used to construct or tune historical labels.

**4. Missing content-level information**

The current five features focus on search performance. They do not directly measure content quality, relevance, freshness, or the actual effect of editing a page.

**5. Data leakage risk**

Same-day clicks and CTR are unsuitable predictors of a same-day label derived from clicks. Features must be available before the intended prediction moment.

**6. Sampling limitations**

The 10,000-row feature frame is a limited development subset selected in a fixed order, not a random representative sample of the full warehouse.

**Conclusion**

The warehouse is suitable for exploratory SEO analysis and feature engineering, but a trustworthy predictive system requires careful target construction, temporal validation, and additional evidence.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.